In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping_update_calo.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

In [ ]:
mc_bnb_hit0_df.columns

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

print("data_tot_pot: %.3e" %(data_tot_pot))
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_pfp_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_pfp_df))

In [ ]:
import pandas as pd

# Define the target 4 index levels identifying unique tracks/slices
target_levels = ['__ntuple', 'entry', 'rec.slc..index', 'rec.slc.reco.pfp..index']

# --- 1. Combined Unique Combinations Across hit0 + hit1 + hit2 ---
hit_dfs = {
    '0': mc_bnb_hit0_df,
    '1': mc_bnb_hit1_df,
    '2': mc_bnb_hit2_df,
}

hit_tuple_sets = []
total_hit_rows = 0

for name, df in hit_dfs.items():
    if not df.empty:
        total_hit_rows += len(df)
        # Extract target 4-tuples as a set
        tuples_set = set(
            df.index.to_frame()[target_levels].itertuples(index=False, name=None)
        )
        hit_tuple_sets.append(tuples_set)

# Union of unique 4-tuples across hit0, hit1, and hit2
if hit_tuple_sets:
    combined_hit_unique_tuples = set.union(*hit_tuple_sets)
    n_unique_combined_hits = len(combined_hit_unique_tuples)
else:
    n_unique_combined_hits = 0

print(f"Combined (hit0+hit1+hit2) total hit rows: {total_hit_rows}")
print(f"Combined (hit0+hit1+hit2) unique 4-tuple combinations: {n_unique_combined_hits}")

print("\n" + "=" * 50 + "\n")

# --- 2. Unique Combinations for mc_bnb_pfp_df ---
mc_bnb_pfp_df = mc_bnb_pfp_df.sort_index(level='__ntuple', ascending=True)

pfp_tuples_set = set(
    mc_bnb_pfp_df.index.to_frame()[target_levels].itertuples(index=False, name=None)
)
n_unique_pfp = len(pfp_tuples_set)

print(f"mc_bnb_pfp_df total rows: {len(mc_bnb_pfp_df)}")
print(f"mc_bnb_pfp_df unique 4-tuple combinations: {n_unique_pfp}")

# --- Optional Check: Overlap between Hits and PFP ---
if n_unique_combined_hits > 0 and n_unique_pfp > 0:
    overlap = len(combined_hit_unique_tuples.intersection(pfp_tuples_set))
    print("\n" + "=" * 50 + "\n")
    print(f"Unique tracks present in BOTH PFP and Hits: {overlap}")

In [ ]:
import pandas as pd

# Define your columns
p_type_col = ('pfp', 'trk', 'truth', 'p', 'p_type', '')
weight_col = ('slc', 'wgt', '', '', '', '')  # Optional: set to None if unweighted

# --- 1. Extract and Clean Data ---
valid_mask = mc_bnb_pfp_df[p_type_col].notna()
df_clean = mc_bnb_pfp_df[valid_mask]

if weight_col and weight_col in df_clean.columns:
    weights = df_clean[weight_col].fillna(1.0)
else:
    weights = pd.Series(1.0, index=df_clean.index)

# --- 2. Calculate Weighted & Unweighted Statistics ---
stats_df = pd.DataFrame({
    'p_type': df_clean[p_type_col],
    'weight': weights
})

summary = stats_df.groupby('p_type').agg(
    Counts=('weight', 'count'),
    Weighted_Yield=('weight', 'sum')
).reset_index()

total_counts = summary['Counts'].sum()
total_weighted = summary['Weighted_Yield'].sum()

summary['Raw_%'] = (summary['Counts'] / total_counts) * 100
summary['Weighted_%'] = (summary['Weighted_Yield'] / total_weighted) * 100

# Sort by weighted yield (descending)
summary = summary.sort_values(by='Weighted_Yield', ascending=False)

# --- 3. Pretty Print Output ---
print("\n" + "="*60)
print(f"{'p_type':<15} | {'Counts':<8} | {'Raw %':<8} | {'Weighted':<10} | {'Weighted %':<10}")
print("-" * 60)

for _, row in summary.iterrows():
    print(f"{str(row['p_type']):<15} | {int(row['Counts']):<8d} | {row['Raw_%']:<7.2f}% | {row['Weighted_Yield']:<10.1f} | {row['Weighted_%']:<9.2f}%")

print("-" * 60)
print(f"{'Total':<15} | {total_counts:<8d} | {100.0:<7.2f}% | {total_weighted:<10.1f} | {100.0:<9.2f}%")
print("="*60 + "\n")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Default bin definitions
DEFAULT_BINS_Y = np.linspace(0, 10, 51)  # dE/dx range [MeV/cm]
DEFAULT_BINS_Z = np.linspace(0, 200, 51) # Residual range [cm]


def plot_split_tpc_2d(
    df: pd.DataFrame,
    x_col: str = "rr",
    y_col: str = "dedx",
    x_split_col: str = "x",
    weight_col: str = None,
    bins_x: np.ndarray = DEFAULT_BINS_Z,
    bins_y: np.ndarray = DEFAULT_BINS_Y,
    xlabel: str = "Residual Range [cm]",
    ylabel: str = "dE/dx [MeV/cm]",
    title_prefix: str = "Hit Distribution",
    cmap_name: str = "viridis",
    figsize: tuple = (15, 6),
):
    """Generates a 1x2 2D histogram multiplot split by X < 0 (left) and X >= 0 (right)."""

    # Clean data & extract arrays safely
    mask = df[x_col].notna() & df[y_col].notna() & df[x_split_col].notna()
    plot_df = df[mask]

    x_vals = plot_df[x_col].values
    y_vals = plot_df[y_col].values
    split_vals = plot_df[x_split_col].values

    if weight_col and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0).values
    else:
        weights = np.ones_like(x_vals)

    finite_mask = (
        np.isfinite(x_vals)
        & np.isfinite(y_vals)
        & np.isfinite(split_vals)
        & np.isfinite(weights)
    )
    x_vals, y_vals, split_vals, weights = (
        x_vals[finite_mask],
        y_vals[finite_mask],
        split_vals[finite_mask],
        weights[finite_mask],
    )

    # Subdivide by TPC side using x_split_col
    mask_neg_x = split_vals < 0
    mask_pos_x = split_vals >= 0

    # Setup 1x2 Subplots with shared Y-axis
    fig, (ax_left, ax_right) = plt.subplots(
        1, 2, figsize=figsize, sharey=True, gridspec_kw={"wspace": 0.08}
    )

    cmap = globals().get("sunset_cmap", cmap_name)

    # Calculate global max for uniform colorbar scaling
    h_left, _, _ = np.histogram2d(
        x_vals[mask_neg_x], y_vals[mask_neg_x], bins=[bins_x, bins_y], weights=weights[mask_neg_x]
    )
    h_right, _, _ = np.histogram2d(
        x_vals[mask_pos_x], y_vals[mask_pos_x], bins=[bins_x, bins_y], weights=weights[mask_pos_x]
    )
    vmax = max(h_left.max(), h_right.max())
    vmax = vmax if vmax > 0 else None

    # --- Left Plot: Split Var < 0 ---
    im0 = ax_left.hist2d(
        x_vals[mask_neg_x],
        y_vals[mask_neg_x],
        bins=[bins_x, bins_y],
        weights=weights[mask_neg_x],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]

    ax_left.set_title(f"{title_prefix}: $X < 0$ cm", fontsize=14, pad=10)
    ax_left.set_xlabel(xlabel, fontsize=14)
    ax_left.set_ylabel(ylabel, fontsize=14)
    ax_left.set_xlim(bins_x[0], bins_x[-1])
    ax_left.set_ylim(bins_y[0], bins_y[-1])
    ax_left.grid(alpha=0.3, linestyle="--")

    # --- Right Plot: Split Var >= 0 ---
    im1 = ax_right.hist2d(
        x_vals[mask_pos_x],
        y_vals[mask_pos_x],
        bins=[bins_x, bins_y],
        weights=weights[mask_pos_x],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]

    ax_right.set_title(f"{title_prefix}: $X \\geq 0$ cm", fontsize=14, pad=10)
    ax_right.set_xlabel(xlabel, fontsize=14)
    ax_right.set_xlim(bins_x[0], bins_x[-1])
    ax_right.grid(alpha=0.3, linestyle="--")

    # Common Colorbar
    cbar = fig.colorbar(im1, ax=[ax_left, ax_right], pad=0.02)
    cbar.set_label("Weighted Entries", fontsize=12)

    return fig, (ax_left, ax_right)

In [ ]:
# Pass plain string column names
for name, hitdf in hit_dfs.items():
    fig, axes = plot_split_tpc_2d(
        df=hitdf,
        x_col="rr",
        y_col="dedx",
        x_split_col="x",
        weight_col=None,
        bins_x=np.linspace(0, 80, 41),   # Residual Range [cm]
        bins_y=np.linspace(0, 10, 41),    # dE/dx [MeV/cm]
        xlabel="Residual Range [cm]",
        ylabel="Hit dE/dx [MeV/cm]",
        title_prefix=f"Plane {name} dE/dx vs RR",
    )

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_dedx_by_rr_and_tpc(
    df: pd.DataFrame,
    rr_range: tuple = (0.0, 1.0),
    dedx_col: str = "dedx",
    rr_col: str = "rr",
    x_col: str = "x",
    weight_col: str = None,
    bins: np.ndarray = np.linspace(0.0, 10.0, 51),
    stacked: bool = False,
    density: bool = False,  # 👈 Added parameter
    alpha: float = 0.3,
    linewidth: float = 1.8,
    figsize: tuple = (8, 6),
    title: str = None,
    ax: plt.Axes = None,
):
    # --- 1. Filter RR Range & Clean Data ---
    mask = (
        df[dedx_col].notna()
        & df[rr_col].notna()
        & df[x_col].notna()
        & (df[rr_col] >= rr_range[0])
        & (df[rr_col] < rr_range[1])
    )
    plot_df = df[mask].copy()

    if plot_df.empty:
        raise ValueError(
            f"No valid entries found for {rr_col} in range {rr_range}."
        )

    # Resolve Weights
    if weight_col is not None and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0)
    else:
        weights = pd.Series(1.0, index=plot_df.index)

    # --- 2. Categorization by X Position ---
    mask_pos_x = plot_df[x_col] > 0
    mask_neg_x = ~mask_pos_x

    grouped_data = [
        plot_df.loc[mask_pos_x, dedx_col],
        plot_df.loc[mask_neg_x, dedx_col],
    ]
    grouped_weights = [
        weights[mask_pos_x],
        weights[mask_neg_x],
    ]

    # --- Handle Normalization (density=True) ---
    bin_width = bins[1] - bins[0]
    if density:
        if stacked:
            # Normalize so the combined stacked area sums to 1.0
            total_weight = sum(w.sum() for w in grouped_weights)
            if total_weight > 0:
                scale = 1.0 / (total_weight * bin_width)
                grouped_weights = [w * scale for w in grouped_weights]
        else:
            # Normalize each subgroup independently so each group's area sums to 1.0
            normed_weights = []
            for w in grouped_weights:
                w_sum = w.sum()
                if w_sum > 0:
                    normed_weights.append(w / (w_sum * bin_width))
                else:
                    normed_weights.append(w)
            grouped_weights = normed_weights

    colors = ["#1f77b4", "#d62728"]  # Blue & Red
    labels = [r"$X > 0$ cm", r"$X \leq 0$ cm"]

    # --- 3. Plotting ---
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()

    # Layer 1: Filled steps
    ax.hist(
        grouped_data,
        bins=bins,
        weights=grouped_weights,
        stacked=stacked,
        histtype="stepfilled",
        color=colors,
        alpha=alpha,
        label=labels,
    )

    # Layer 2: Outlines
    ax.hist(
        grouped_data,
        bins=bins,
        weights=grouped_weights,
        stacked=stacked,
        histtype="step",
        color=colors,
        linewidth=linewidth,
    )

    # --- 4. Styling & Formatting ---
    ax.set_xlabel(r"Hit $dE/dx$ [MeV/cm]", fontsize=14)

    if density:
        ax.set_ylabel("A.U.", fontsize=14)
    elif weight_col:
        ax.set_ylabel("Weighted Hits", fontsize=14)
    else:
        ax.set_ylabel("Hits", fontsize=14)

    if title is None:
        title = rf"Hit $dE/dx$ Distribution ({rr_range[0]} $\leq$ RR < {rr_range[1]} cm)"
    ax.set_title(title, fontsize=15, pad=12)

    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

    ax.tick_params(axis="both", which="both", labelsize=12, direction="in")
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

    ax.legend(
        fontsize=12,
        frameon=True,
        framealpha=1.0,
        edgecolor="black",
        fancybox=False,
    )

    return fig, ax

In [ ]:
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(2, 3), (6, 7), (15, 16)]

for rr_min, rr_max in rr_ranges:
    fig, axes = plt.subplots(
        1, len(hit_dfs), figsize=(18, 5), sharey=True, gridspec_kw={"wspace": 0.08}
    )

    max_y_value = 0  # Track global maximum density across planes

    for plane_idx, (df_plane, ax) in enumerate(zip(hit_dfs, axes)):
        try:
            plot_dedx_by_rr_and_tpc(
                df=df_plane,
                rr_range=(rr_min, rr_max),
                dedx_col="dedx",
                rr_col="rr",
                x_col="x",
                weight_col=None,
                bins=np.linspace(0.0, 10.0, 51),
                stacked=False,
                density=True,  # 👈 Now supported!
                title=f"Plane {plane_idx}",
                ax=ax,
            )

            # Record maximum bin height in this subplot
            current_max = ax.get_ylim()[1] / 1.15
            if current_max > max_y_value:
                max_y_value = current_max

        except ValueError:
            ax.set_title(f"Plane {plane_idx}: No Data", fontsize=15, pad=12)
            continue

        # Clean up side subplots
        if plane_idx > 0:
            ax.set_ylabel("")
            legend = ax.get_legend()
            if legend:
                legend.remove()

    # Apply the global max limit + 15% headroom to all subplots
    if max_y_value > 0:
        axes[0].set_ylim(0, max_y_value * 1.15)

    fig.suptitle(
        rf"Normalized Hit $dE/dx$ Comparison ({rr_min} $\leq$ RR < {rr_max} cm)",
        fontsize=16,
        y=1.03,
    )

    plt.show()

In [ ]:
hfit = load_physics_classes()

In [ ]:
pdg = 13
particle="muon"
if "pion" in bnb_path:
    pdg = 211
    particle = "pion"
    
theoretical_mpv = make_theoretical_mpv_func(hfit, pdg=pdg) 
all_results, fit_params = analyze(hit_dfs, particle=particle,
                                  theoretical_mpv_func=theoretical_mpv)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Style mapping for consistency across both functions
TPC_STYLES = {
    0: {"linestyle": "--", "marker": "o", "label_prefix": "x < 0 (TPC 0)"},
    1: {"linestyle": ":", "marker": "s", "label_prefix": "x > 0 (TPC 1)"},
    -1: {"linestyle": "-", "marker": "^", "label_prefix": "Combined"},
}


def plot_all_planes(all_results, fit_params, particle, out_prefix):
    """Plots all (plane, tpc) power-law fit curves on a single square plot."""
    fig, ax = plt.subplots(figsize=(8, 8))

    for (plane, tpc), res in all_results.items():
        popt, _ = fit_params.get((plane, tpc), (None, None))
        if popt is None or res is None or not len(res):
            continue

        color = PLANE_COLORS[plane]
        style_info = TPC_STYLES.get(
            tpc,
            {"linestyle": "-", "label_prefix": f"tpc={tpc}"},
        )
        style = style_info["linestyle"]
        prefix = style_info["label_prefix"]

        xs = np.linspace(res["mpv_x"].min(), res["mpv_x"].max(), 200)
        label = (
            f"{prefix}, Plane {plane}: {popt[0]:.2f} + {popt[1]:.2f}x^{popt[2]:.2f}"
        )
        ax.plot(
            xs,
            power_law(xs, *popt),
            linestyle=style,
            color=color,
            linewidth=2.0,
            label=label,
        )

    ax.set_xlabel("MPV [MeV/cm]", fontsize=12)
    ax.set_ylabel(r"$\sigma_G$ [MeV/cm]", fontsize=12)

    # Enlarged legend font size
    ax.legend(fontsize=10, loc="upper left", framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.5)

    # Force square plot aspect ratio
    ax.set_box_aspect(1)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_planes.pdf")
    plt.show()


import matplotlib.pyplot as plt
import numpy as np

# Style mapping for consistency
TPC_STYLES = {
    0: {"linestyle": "--", "marker": "o", "label_prefix": "x < 0 (TPC 0)"},
    1: {"linestyle": ":", "marker": "s", "label_prefix": "x > 0 (TPC 1)"},
    -1: {"linestyle": "-", "marker": "^", "label_prefix": "Combined"},
}


def plot_by_plane(
    all_results,
    fit_params,
    particle="muon",
    out_prefix="comparison",
    x_limits=None,  # Defaults to full range (None)
):
    """One square subplot per plane with TPC 0, 1, and Combined overlaid."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    tpc_colors = {0: "tab:red", 1: "tab:blue", -1: "tab:green"}

    for plane in range(3):
        ax = axes[plane]
        y_visible_min, y_visible_max = np.inf, -np.inf

        for tpc in [0, 1, -1]:
            res = all_results.get((plane, tpc))
            popt, _ = fit_params.get((plane, tpc), (None, None))

            if res is None or not len(res):
                continue

            color = tpc_colors[tpc]
            style_info = TPC_STYLES[tpc]
            style = style_info["linestyle"]
            marker = style_info["marker"]
            prefix = style_info["label_prefix"]

            # Filter data points if x_limits is provided
            if x_limits is not None:
                mask = (res["mpv_x"] >= x_limits[0]) & (
                    res["mpv_x"] <= x_limits[1]
                )
                res_filtered = res[mask]
            else:
                res_filtered = res

            # Plot data points
            if len(res_filtered):
                ax.errorbar(
                    res_filtered["mpv_x"],
                    res_filtered["gsigma"],
                    yerr=res_filtered["gsigma_err"],
                    fmt=marker,
                    color=color,
                    ms=4,
                    capsize=2,
                    label=f"{prefix} Data",
                )
                if x_limits is not None:
                    y_visible_min = min(
                        y_visible_min,
                        (
                            res_filtered["gsigma"] - res_filtered["gsigma_err"]
                        ).min(),
                    )
                    y_visible_max = max(
                        y_visible_max,
                        (
                            res_filtered["gsigma"] + res_filtered["gsigma_err"]
                        ).max(),
                    )

            # Plot fit curve
            if popt is not None:
                x_min = x_limits[0] if x_limits else res["mpv_x"].min()
                x_max = x_limits[1] if x_limits else res["mpv_x"].max()
                xs = np.linspace(x_min, x_max, 200)
                ys = power_law(xs, *popt)

                label_fit = f"{prefix}: {popt[0]:.2f} + {popt[1]:.2f}x^{popt[2]:.2f}"
                ax.plot(
                    xs,
                    ys,
                    linestyle=style,
                    color=color,
                    linewidth=1.8,
                    label=label_fit,
                )

                if x_limits is not None:
                    y_visible_min = min(y_visible_min, ys.min())
                    y_visible_max = max(y_visible_max, ys.max())

        # Apply zoom limits and adjust Y limits only when x_limits is explicitly specified
        if x_limits is not None:
            ax.set_xlim(x_limits)
            if not np.isinf(y_visible_min) and not np.isinf(y_visible_max):
                y_pad = (y_visible_max - y_visible_min) * 0.1
                ax.set_ylim(y_visible_min - y_pad, y_visible_max + y_pad)

        ax.set_title(f"Plane {plane}", fontsize=14, pad=10)
        ax.set_xlabel("MPV [MeV/cm]", fontsize=12)
        ax.set_ylabel(r"$\sigma_G$ [MeV/cm]", fontsize=12)

        ax.legend(fontsize=9, loc="upper left", framealpha=0.9)
        ax.grid(True, linestyle=":", alpha=0.5)

        # Enforce 1:1 square aspect ratio
        ax.set_box_aspect(1)

    fig.tight_layout()
    zoom_suffix = (
        f"_zoom_{x_limits[0]}_{x_limits[1]}".replace(".", "p")
        if x_limits
        else ""
    )
    fig.savefig(f"{out_prefix}_by_plane{zoom_suffix}.pdf")
    plt.show()

In [ ]:
plot_all_planes(
    all_results,
    fit_params,
    particle=particle,
    out_prefix="comparison",
)

plot_by_plane(
    all_results,
    fit_params,
    particle=particle,
    out_prefix="comparison"
)


plot_by_plane(
    all_results,
    fit_params,
    particle=particle,
    out_prefix="comparison",
    x_limits= [1.9, 3]
)

In [ ]:
def plot_slice_diagnostic(
    df,
    plane,
    tpc,
    rr_min,
    rr_max,
    hfit=None,
    pdg=13,
    mass=None,
    fit_params=None,
    dedx_col="dedx",
    nbins=100,
    hist_xmin=0.0,
    hist_xmax=10.0,
    first_stage_range=(0.0, 10.0),
):
    """Sanity-check plot for a single (plane, tpc, rr-range) slice: the raw dE/dx

    histogram, the two-stage Langau fit from section 3, and (if `hfit` is given)
    the theoretical PDF from `build_theoretical_pdf`, all on one axes. Useful for
    eyeballing a handful of slices before running the full `analyze()` loop
    over every rr bin.

    If `fit_params` is also given (the dict returned by `analyze()`, i.e.
    `fit_params[(plane, tpc)] = (popt, perr)` for the sigma_G(MPV) power
    law from `fit_sigma_vs_mpv`), two more curves are added: the theoretical
    PDF convolved with the Gaussian-smearing sigma that power law
    *predicts* at this slice's theoretical MPV, both unshifted (raw
    convolution, mode moves with the skew) and recentered-to-mode (peak
    forced back to the unconvolved PDF's peak). Comparing the two directly
    shows how much of the mismatch between the smeared prediction and the
    data is coming from the mode-shift itself vs. everything else. Skipped
    if `fit_params` has no entry for this (plane, tpc) (e.g. too few good
    slices to fit there).

    The Langau-fit curve and all theory curves are each rescaled so their
    peak matches the data histogram's peak -- their "natural" normalizations
    differ (the fit is in raw counts, the theory curves are unit-area PDFs
    times an approximate norm factor), so without this they can come out
    much taller/shorter than the data even when the underlying shape
    agrees. Rescaling to a common peak height makes this a pure shape
    comparison, and the shared height also means one y-limit covers
    everything.
    """
    sl = df if tpc == -1 else df[df["tpc"] == tpc]
    sl = sl[
            (sl["rr"] >= rr_min) & (sl["rr"] < rr_max) & (sl["pitch"] <= 2)
    ]
    rr_center = 0.5 * (rr_min + rr_max)

    fig, ax = plt.subplots(figsize=(8, 5.5))
    counts, edges, _ = ax.hist(
        sl[dedx_col].dropna(),
        bins=nbins,
        range=(hist_xmin, hist_xmax),
        histtype="stepfilled",
        alpha=0.3,
        color="steelblue",
        edgecolor="navy",
        label=f"data (N={len(sl)})",
    )
    bin_width = (hist_xmax - hist_xmin) / nbins
    x_eval = np.linspace(hist_xmin, hist_xmax, 400)
    data_max = counts.max() if len(counts) else 0.0
    curve_max = data_max  # track the overall peak across everything plotted

    hist = th1_from_series(
        sl[dedx_col],
        f"diag_p{plane}_t{tpc}",
        "",
        nbins,
        hist_xmin,
        hist_xmax,
    )
    fit = langau_fit_two_stage(hist, first_stage_range=first_stage_range)
    if fit is not None:
        width, mpv_reco, area, gsigma = fit["pars"]
        f_curve = ROOT.TF1(
            f"diag_curve_p{plane}_t{tpc}",
            ROOT.langau_cpp,
            hist_xmin,
            hist_xmax,
            4,
        )
        f_curve.SetParameters(width, mpv_reco, area, gsigma)
        y_fit = np.array([f_curve.Eval(xv) for xv in x_eval])
        if y_fit.max() > 0 and data_max > 0:
            y_fit = y_fit / y_fit.max() * data_max
        ax.plot(
            x_eval,
            y_fit,
            "g--",
            lw=2,
            label=f"Langau fit (MPV={mpv_reco:.2f})",
        )
        curve_max = max(curve_max, y_fit.max())

    if hfit is not None:
        mean_pitch = sl["pitch"].median() if "pitch" in sl else 0.3
        #mean_pitch = 0.32
        pdf = build_theoretical_pdf(
            hfit, pdg, rr_center, mean_pitch, mass=mass
        )
        theo_mpv = pdf.GetMaximumX()
        norm = len(sl) * bin_width
        y_theo = np.array([pdf.Eval(xv) * norm for xv in x_eval])
        if y_theo.max() > 0 and data_max > 0:
            y_theo = y_theo / y_theo.max() * data_max
        ax.plot(
            x_eval,
            y_theo,
            "r:",
            lw=2.5,
            label=f"theory, no smearing (MPV={theo_mpv:.2f})",
        )
        curve_max = max(curve_max, y_theo.max())

        popt = None
        if fit_params is not None:
            popt, _ = fit_params.get((plane, tpc), (None, None))
        if popt is not None:
            predicted_sigma_g = power_law(theo_mpv, *popt)

            # --- Unshifted convolution: raw mode-shift left in, same as
            # the ROOT macro's f0 (before the dx correction). ---
            y_pred_unshifted = convolve_pdf_with_gaussian(
                pdf, predicted_sigma_g, x_eval, recenter_to_mode=False
            )
            if y_pred_unshifted.max() > 0 and data_max > 0:
                y_pred_unshifted = y_pred_unshifted / y_pred_unshifted.max() * data_max
            ax.plot(
                x_eval,
                y_pred_unshifted,
                color="mediumpurple",
                linestyle="--",
                lw=2.0,
                label=rf"theory $\otimes$ resolution, unshifted",
            )
            curve_max = max(curve_max, y_pred_unshifted.max())

            # --- Recentered convolution: peak forced back to the
            # unconvolved PDF's mode, same as the ROOT macro's f_shifted. ---
            y_pred_shifted = convolve_pdf_with_gaussian(
                pdf, predicted_sigma_g, x_eval, recenter_to_mode=True
            )
            if y_pred_shifted.max() > 0 and data_max > 0:
                y_pred_shifted = y_pred_shifted / y_pred_shifted.max() * data_max
            ax.plot(
                x_eval,
                y_pred_shifted,
                color="darkorange",
                linestyle="-",
                lw=2.5,
                label=rf"theory $\otimes$ predicted resolution, shifted ($\sigma_G$={predicted_sigma_g:.3f})",
            )
            curve_max = max(curve_max, y_pred_shifted.max())

    ax.set_ylim(0, curve_max * 1.15 if curve_max > 0 else 1.0)
    ax.set_xlabel(r"$dE/dx$ [MeV/cm]")
    ax.set_ylabel(f"hits / {bin_width:.2f} MeV/cm")
    ax.set_title(f"plane {plane}, tpc {tpc}, {rr_min:g} <= rr < {rr_max:g} cm")
    if tpc == -1:
        ax.set_title(
            f"plane {plane}, TPCs Combined, {rr_min:g} <= rr < {rr_max:g} cm"
        )
          
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()
    return fig

In [ ]:
print(df["pitch"].mean())

In [ ]:
fit_params

In [ ]:
fit_params_2 = {
    # Plane 0
    (0, 0): ([0.127705, 0.0260447, 2.05065], None),
    (0, 1): ([0.127705, 0.0260447, 2.05065], None),
    (0, -1): ([0.127705, 0.0260447, 2.05065], None),
    # Plane 1
    (1, 0): ([0.0489597, 0.0926731, 1.32994], None),
    (1, 1): ([0.0489597, 0.0926731, 1.32994], None),
    (1, -1): ([0.0489597, 0.0926731, 1.32994], None),
    # Plane 2
    (2, 0): ([0.0339961, 0.014597, 2.49474], None),
    (2, 1): ([0.0339961, 0.014597, 2.49474], None),
    (2, -1): ([0.0339961, 0.014597, 2.49474], None),
}

In [ ]:
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(1, 2), (2, 3), (3,4), (4, 5), (6, 7), (15, 16), (30, 31)]

import os
# Define output path
output_dir = "/exp/sbnd/data/users/lpelegri/TLEGraphs/dEdxPDF"
os.makedirs(output_dir, exist_ok=True)

plane = 2
df =  hit_dfs[plane]

for rr_min, rr_max in rr_ranges:
    # 1. Full Diagnostic Plot
    fig1 = plot_slice_diagnostic(
        df,
        plane,
        -1,
        rr_min,
        rr_max,
        hfit=hfit,
        pdg=pdg,
        fit_params=fit_params,
        dedx_col="dedx",
        nbins=100,
        hist_xmin=0.0,
        hist_xmax=10.0,
    )

    fname1 = f"diag_full_p2_tpc0_rr_{rr_min:g}_{rr_max:g}.png"
    fig1.savefig(
        os.path.join(output_dir, fname1), dpi=300, bbox_inches="tight"
    )

    # 2. Theory-Only Plot
    fig2 = plot_slice_diagnostic_theory_only(
        df,
        plane,
        -1,
        rr_min,
        rr_max,
        hfit=hfit,
        pdg=pdg,
        fit_params=fit_params,
        dedx_col="dedx",
        nbins=100,
        hist_xmin=0.0,
        hist_xmax=10.0,
    )

    fname2 = f"diag_theory_p2_tpc0_rr_{rr_min:g}_{rr_max:g}.png"
    fig2.savefig(
        os.path.join(output_dir, fname2), dpi=300, bbox_inches="tight"
    )
    

In [ ]:
import matplotlib.pyplot as plt

# Filter pitch between 0 and 4
filtered_pitch = df[(df["pitch"] >= 0) & (df["pitch"] <= 2)]["pitch"].dropna()

plt.figure(figsize=(8, 5))
plt.hist(
    filtered_pitch,
    bins=50,
    range=(0, 4),  # Forces bin boundaries strictly within [0, 4]
    color="#1f77b4",
    edgecolor="black",
    alpha=0.7,
)

plt.xlabel("Pitch [cm]", fontsize=12)
plt.ylabel("Counts", fontsize=12)
plt.title("Distribution of Hit Pitch Values", fontsize=14)
plt.xlim(0, 2)  # Sets axis limits strictly from 0 to 4
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

# Fit with no plane distinction

In [ ]:
def _parabolic_peak(grid, vals):
    """Refines an argmax peak location with a 3-point parabolic interpolation,
    giving sub-grid-step precision instead of snapping to the nearest grid point."""
    i = np.argmax(vals)
    if i == 0 or i == len(vals) - 1:
        return grid[i]  # can't interpolate at the edge, fall back to raw argmax
    y0, y1, y2 = vals[i - 1], vals[i], vals[i + 1]
    denom = (y0 - 2 * y1 + y2)
    if denom == 0:
        return grid[i]
    # vertex of the parabola through the 3 points, in units of grid spacing
    delta = 0.5 * (y0 - y2) / denom
    step = grid[i + 1] - grid[i]
    return grid[i] + delta * step


def get_convolution_shift(pdf, sigma_g, n_sigma=5.0, grid_step=None, grid_range=(-10.0, 20.0)):
    if sigma_g is None or not np.isfinite(sigma_g) or sigma_g <= 0:
        return None, None, None

    if grid_step is None:
        grid_step = sigma_g / 25.0

    n_pad = int(np.ceil((n_sigma * sigma_g) / grid_step))
    pad_width = n_pad * grid_step

    grid = np.arange(grid_range[0] - pad_width, grid_range[1] + pad_width + grid_step, grid_step)
    pdf_vals = np.array([pdf.Eval(t) for t in grid])

    k = np.arange(-n_pad, n_pad + 1)
    kernel_x = k * grid_step
    kernel = np.exp(-0.5 * (kernel_x / sigma_g) ** 2)
    kernel /= kernel.sum()

    conv_vals = np.convolve(pdf_vals, kernel, mode="same")

    mode_pdf = _parabolic_peak(grid, pdf_vals)
    mode_conv = _parabolic_peak(grid, conv_vals)
    shift = mode_conv - mode_pdf

    return mode_pdf, mode_conv, shift


def compute_shift_vs_rr(
    df,
    plane,
    tpc,
    hfit,
    fit_params,
    pdg=13,
    mass=None,
    rr_min=3.0,
    rr_max=40.0,
    rr_bin_width=1.0,
    min_entries=30,
    verbose=True,
):
    """Loops rr slices (rr_min, rr_min+1) ... up to rr_max, and for each
    computes the theoretical MPV, the predicted Gaussian-smearing sigma
    (from fit_params[(plane, tpc)]'s power law), and the resulting
    convolution-induced peak shift.

    Returns a DataFrame with columns: rr_min, rr_max, rr_center, mean_pitch,
    theo_mpv, predicted_sigma_g, mode_pdf, mode_conv, shift.
    """
    popt, _ = fit_params.get((plane, tpc), (None, None))
    if popt is None:
        raise ValueError(
            f"No sigma_G(MPV) power-law fit found for (plane={plane}, tpc={tpc}) "
            "in fit_params -- run analyze() (or fit_sigma_vs_mpv) for this key first."
        )

    sl_all = df if tpc == -1 else df[df["tpc"] == tpc]

    edges = make_rr_edges(rr_min, rr_max, rr_bin_width)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        rr_center = 0.5 * (lo + hi)

        sl = sl_all[(sl_all["rr"] >= lo) & (sl_all["rr"] < hi) & (sl_all["pitch"] <= 2)]
        if len(sl) < min_entries:
            if verbose:
                print(f"  rr=[{lo:g},{hi:g}): skipped, only {len(sl)} hits (< {min_entries})")
            continue

        mean_pitch = sl["pitch"].median()

        pdf = build_theoretical_pdf(hfit, pdg, rr_center, mean_pitch, mass=mass)
        theo_mpv = pdf.GetMaximumX()

        predicted_sigma_g = power_law(theo_mpv, *popt)

        mode_pdf, mode_conv, shift = get_convolution_shift(pdf, predicted_sigma_g)
        if shift is None:
            if verbose:
                print(f"  rr=[{lo:g},{hi:g}): skipped, invalid predicted_sigma_g={predicted_sigma_g}")
            continue

        rows.append(dict(
            rr_min=lo, rr_max=hi, rr_center=rr_center,
            mean_pitch=mean_pitch,
            theo_mpv=theo_mpv,
            predicted_sigma_g=predicted_sigma_g,
            mode_pdf=mode_pdf,
            mode_conv=mode_conv,
            shift=shift,
        ))

    return pd.DataFrame(rows)


def plot_shift_vs_rr(shift_df, plane, tpc, particle="muon", out_prefix="langau_rr"):
    """Plots the convolution-induced peak shift as a function of rr-slice center."""
    fig, ax1 = plt.subplots(figsize=(9, 5.5))

    ax1.plot(shift_df["rr_center"], shift_df["shift"], "o-", color="#d62728", label="peak shift")
    ax1.axhline(0, color="gray", linewidth=1, linestyle=":")
    ax1.set_xlabel("rr slice center [cm]", fontsize=12)
    ax1.set_ylabel(r"shift = mode(conv) $-$ mode(PDF)  [MeV/cm]", fontsize=12, color="#d62728")
    ax1.tick_params(axis="y", labelcolor="#d62728")
    ax1.grid(True, linestyle=":", alpha=0.5)

    # Overlay predicted_sigma_g on a second axis for context -- the shift
    # should scale roughly with sigma_g, so it's useful to see them together.
    ax2 = ax1.twinx()
    ax2.plot(shift_df["rr_center"], shift_df["predicted_sigma_g"], "s--", color="#1f77b4", alpha=0.6, label=r"predicted $\sigma_G$")
    ax2.set_ylabel(r"predicted $\sigma_G$ [MeV/cm]", fontsize=12, color="#1f77b4")
    ax2.tick_params(axis="y", labelcolor="#1f77b4")

    tpc_label = "Combined" if tpc == -1 else f"TPC {tpc}"
    ax1.set_title(f"Convolution-induced peak shift — {particle.capitalize()}, Plane {plane}, {tpc_label}", fontsize=13)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_shift_vs_rr_p{plane}_tpc{tpc}.pdf")
    plt.show()
    return fig


# --- Driver ---
shift_df = compute_shift_vs_rr(
    hit_dfs[2],      # plane-2 dataframe -- swap for whichever plane you're checking
    plane=2,
    tpc=1,
    hfit=hfit,
    fit_params=fit_params,
    pdg=pdg,
    rr_min=3.0,
    rr_max=100.0,
    rr_bin_width=1.0,
)
plot_shift_vs_rr(shift_df, plane=2, tpc=1, particle="muon", out_prefix=os.path.join(output_dir, "langau_rr"))